# 02 - Creating New Optimized Versions of High-Value NASA Datasets

Below we build upon the work in check-co-hdf5-cmr.ipynb to build optimized versions of the data and a config file for testing with libraries. 

* repacked
* kerchunk original
* kerchunk repacked

To simplify this evaluation, we only use datasets which VEDA Hub has access to (which is most of the most highly used data sets).

Datasets skipped:
* if fsm_strategy and page_size were not found, likely this collection is not made up of valid HDF5 files
* oco2_L1aInSB_00018a_140703_B11000r_220105200708.h5 - compled data structure, not sure where to find data variables
* VNP03IMG.A2012019.0000.002.2020318135750.nc - don't have access to this bucket yet

In [1]:
import boto3
import fsspec
import inspect
import json
from kerchunk.hdf import SingleHdf5ToZarr
import pandas as pd
import subprocess
import yaml

# Define reprocessing functions

In [2]:
fs_read = fsspec.filesystem("s3", anon=False)

def generate_json_reference(in_filename, out_filename):
    so = dict(mode="rb", default_fill_cache=False, default_cache_type="first")
    with fs_read.open(in_filename, **so) as infile:     
        h5chunks = SingleHdf5ToZarr(in_filename)
        suffix = out_filename.split(".")[-1]
        out_filename = out_filename.replace(suffix, 'json')
        with open(out_filename, 'w') as outfile:
            outfile.write(json.dumps(h5chunks.translate()))
        return out_filename

In [3]:
processing_options = {
    "original": {
        "procssing": None,
        "input_file": None,
        "link": None
    },
    "repacked_page_4mb": {
        "processing": "h5repack -S PAGE -G 4000000",
        "input_file": "original",
        "link": None
    },   
    "kerchunk": {
        "processing": generate_json_reference,
        "input_file": "original",
        "link": None
    },
    "kerchunk_repacked_page_4mb": {
        "processing": generate_json_reference,
        "input_file": "repacked_page_4mb",
        "link": None
    }
}

# Open file with HDF5/NetCDF-4 collections

In [4]:
data_json = json.loads(open('hdf_cmr_query_results.json', 'r').read())
# remove some which had errors
filtered_list = [list(item.values())[0] for item in data_json if isinstance(item, dict)]
df = pd.DataFrame(data=filtered_list)
df = df.set_index('filename')

In [5]:
df[20:40]

,collection,version,direct_link,fsm_strategy,page_size
filename,,,,,
AIRS.2010.01.01.001.L2.CO2_Std_IR.v5.4.11.0.CO2.T12178090647.hdf,AIRS2STC,005,s3://gesdisc-cumulus-prod-protected/Aqua_AIRS_...,Not found,Not found
OMI-Aura_L2-OMTO3_2004m1001t0003-o01132_v003-2012m0330t162838.he5,OMTO3,003,s3://gesdisc-cumulus-prod-protected/Aura_OMI_L...,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
SNDR.J1.CRIMSS.20180217T0000.m06.g001.L2_CLIMCAPS_RET.std.v02_28.G.200221153452.nc,SNDRJ1IML2CCPRET,2,s3://gesdisc-cumulus-prod-protected/JPSS1_Soun...,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
GSSTF_NCEP.3.1987.07.01.he5,GSSTF_NCEP,3,s3://gesdisc-cumulus-prod-protected/GSSTF/GSST...,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
MOD09GA.A2000055.h25v08.061.2020037171358.hdf,MOD09GA,061,s3://lp-prod-protected/MOD09GA.061/MOD09GA.A20...,Not found,Not found
MOD09GQ.A2000055.h32v07.061.2020037171929.hdf,MOD09GQ,061,s3://lp-prod-protected/MOD09GQ.061/MOD09GQ.A20...,Not found,Not found
LPRM-AMSR2_L3_D_SOILM3_V001_20120703005731.nc4,LPRM_AMSR2_D_SOILM3,001,s3://gesdisc-cumulus-prod-protected/WAOB/LPRM_...,H5F_FSPACE_STRATEGY_FSM_AGGR,4096
MYD09GA.A2002185.h24v15.061.2020071172856.hdf,MYD09GA,061,s3://lp-prod-protected/MYD09GA.061/MYD09GA.A20...,Not found,Not found
OMI-Aura_L2-OMCLDRR_2004m1001t0003-o01132_v003-2018m0420t181201.he5,OMCLDRR,003,s3://gesdisc-cumulus-prod-protected/Aura_OMI_L...,H5F_FSPACE_STRATEGY_FSM_AGGR,4096


In [6]:
file_key = 'ATL03_20181014000347_02350101_006_02.h5'
file_data = df.loc[file_key]
short_name, version, direct_link = file_data.collection, file_data.version, file_data.direct_link
processing_options["original"]["link"] = direct_link

# Reprocess and upload

For each reprocessing option, run the reprocessing option on the input file and then upload the file and it's processing to S3.

In [7]:
destination_bucket_name = 'nasa-veda-scratch'
destination_directory = f'eodc_hdf5_experiments/{short_name}___{version}'
s3_resource = boto3.resource('s3')
s3_client = boto3.client('s3')
for processing_option, processing_data in processing_options.items():
    if processing_option == 'original':   
        # Just copy the file
        print(f"Uploading original to s3://{destination_bucket_name}/{destination_directory}/original/{file_key}")
        source_bucket = direct_link.split("/")[2]
        copy_source = {
            'Bucket': source_bucket,
            'Key': direct_link.split(f"{source_bucket}/")[-1]
        } 
        s3_client.copy(copy_source, destination_bucket_name, f"{destination_directory}/original/{file_key}")
        processing_options['original']['link'] = f"s3://{destination_bucket_name}/{destination_directory}/original/{file_key}"
    else:
        processing_cmd = processing_data['processing']
        output_file = f"{processing_option}___{file_key}"
        input_file = processing_options[processing_data["input_file"]]["link"]
        print(f"Processing {processing_cmd} on {input_file}")
        if type(processing_cmd) == str:
            cp_result = subprocess.run(["aws", "s3", "cp", direct_link, file_key], capture_output=True, text=True, check=True)
            processing_cmd_str = f"{processing_cmd} {file_key} {output_file}"
            process_result = subprocess.run(processing_cmd_str.split(" "), capture_output=True, text=True)
            # Check if there was an error
            if process_result.returncode != 0:
                print("Error:", process_result.stderr)            
        elif callable(processing_cmd) == True:
            output_file = processing_cmd(input_file, output_file)
            processing_cmd_str = inspect.getsource(processing_cmd)
            # set it to a string so we can use it for test configuration later
            processing_options[processing_option]['processing'] = processing_cmd_str
        print(f"Processing complete, uploading file to s3://{destination_bucket_name}/{destination_directory}/{output_file}")
        destination_bucket = s3_resource.Bucket(destination_bucket_name)
        destination_bucket.upload_file(output_file,  f"{destination_directory}/{output_file}")
        processing_options[processing_option]["link"] = f"s3://{destination_bucket_name}/{destination_directory}/{output_file}"
        print(f"Uploading processing cmd to s3://{destination_bucket_name}/{destination_directory}/{processing_option}_processing.txt\n")
        s3_client.put_object(
            Bucket=destination_bucket_name,
            Key=f"{destination_directory}/{processing_option}_processing.txt",
            Body=processing_cmd_str
        )

Uploading original to s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/original/ATL03_20181014000347_02350101_006_02.h5
Processing h5repack -S PAGE -G 4000000 on s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/original/ATL03_20181014000347_02350101_006_02.h5
Processing complete, uploading file to s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/repacked_page_4mb___ATL03_20181014000347_02350101_006_02.h5
Uploading processing cmd to s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/repacked_page_4mb_processing.txt

Processing <function generate_json_reference at 0x7f68eb4e8d60> on s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/original/ATL03_20181014000347_02350101_006_02.h5
Processing complete, uploading file to s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/kerchunk___ATL03_20181014000347_02350101_006_02.json
Uploading processing cmd to s3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/kerchunk_processing.txt

Processing <funct

In [8]:
processing_options

{'original': {'procssing': None,
  'input_file': None,
  'link': 's3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/original/ATL03_20181014000347_02350101_006_02.h5'},
 'repacked_page_4mb': {'processing': 'h5repack -S PAGE -G 4000000',
  'input_file': 'original',
  'link': 's3://nasa-veda-scratch/eodc_hdf5_experiments/ATL03___006/repacked_page_4mb___ATL03_20181014000347_02350101_006_02.h5'},
 'kerchunk': {'processing': 'def generate_json_reference(in_filename, out_filename):\n    so = dict(mode="rb", default_fill_cache=False, default_cache_type="first")\n    with fs_read.open(in_filename, **so) as infile:     \n        h5chunks = SingleHdf5ToZarr(in_filename)\n        suffix = out_filename.split(".")[-1]\n        out_filename = out_filename.replace(suffix, \'json\')\n        with open(out_filename, \'w\') as outfile:\n            outfile.write(json.dumps(h5chunks.translate()))\n        return out_filename\n',
  'input_file': 'original',
  'link': 's3://nasa-veda-scratch/eodc_hdf

# Find a group and variable to test

In [9]:
import h5py
h5file = h5py.File(file_key)

In [10]:
# tunnel down through 
h5file.keys()

<KeysViewHDF5 ['METADATA', 'ancillary_data', 'atlas_impulse_response', 'ds_surf_type', 'ds_xyz', 'gt1r', 'gt2r', 'gt3r', 'orbit_info', 'quality_assessment']>

In [11]:
group = '/gt1r/heights'
variable = 'h_ph'

# Create tests object

In [12]:
test_dict = {
    "collection": short_name,
    "group": group,
    "variable": variable,
    "files": processing_options
}

In [13]:
# Write the dictionary to a file
with open (f'h5cloud/h5cloud/file_configs/{short_name}_{version}.yaml', 'w') as file:
    yaml.dump(test_dict, file, default_flow_style=False)


In [14]:
!ls h5cloud/h5cloud/file_configs/

ATL03_006.yaml	       GSSTF_NCEP_3.yaml	     OMVFPITMET_003.yaml
ATL08_006.yaml	       LPRM_AMSR2_D_SOILM3_001.yaml  SNDRSNIML2CCPRETN_2.yaml
GPM_3IMERGHHE_06.yaml  OMUFPITMET_003.yaml	     atl03.yml
